### Setup of timesfm library.
- pip installation of requirements
- Then restart the runtime.

In [2]:
# Clone the repository
!git clone https://github.com/google-research/timesfm.git
%cd timesfm

# Append the finetuning package to the packages list in pyproject.toml.
# This command finds the line starting with "packages = [" and replaces the trailing "]"
# with ', { include = "finetuning", from = "src" }]' to add the finetuning package.
!sed -i '/^packages = \[/ s/]/, { include = "finetuning", from = "src" }]/' pyproject.toml

# Install the package in editable mode
!pip install -e .

fatal: destination path 'timesfm' already exists and is not an empty directory.
/content/timesfm
Obtaining file:///content/timesfm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for timesfm (pyproject.toml) ... done
  Created wheel for timesfm: filename=timesfm-1.2.8-py3-none-any.whl size=10041 sha256=eb8dae3969f5185986bc7ecb09ddf804cc39c690f6f861e44291b43e3adf7371
  Stored in directory: /tmp/pip-ephem-wheel-cache-9_2rqdi_/wheels/61/fc/68/8410c5d0ea03d175d2d18c29227b259d8624df1128d5455b14
Successfully built timesfm
  Attempting uninstall: timesfm
    Found existing installation: timesfm 1.2.8
    Uninstalling timesfm-1.2.8:
      Successfully uninstalled timesfm-1.2.8


In [3]:
!pip show timesfm

Name: timesfm
Version: 1.2.8
Summary: Open weights time-series foundation model from Google Research.
Home-page: https://github.com/google-research/timesfm
Author: Rajat Sen
Author-email: senrajat@google.com
License: 
Location: /usr/local/lib/python3.11/dist-packages
Editable project location: /content/timesfm
Requires: absl-py, einshape, huggingface_hub, numpy, pandas, scikit-learn, typer, utilsforecast, wandb
Required-by: 


####Imports

In [4]:
from os import path
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.multiprocessing as mp
import yfinance as yf
from finetuning.finetuning_torch import FinetuningConfig, TimesFMFinetuner
from huggingface_hub import snapshot_download
from torch.utils.data import Dataset

from timesfm import TimesFm, TimesFmCheckpoint, TimesFmHparams
from timesfm.pytorch_patched_decoder import PatchedTimeSeriesDecoder
import os

# additional import
from numpy.typing import ArrayLike
import kagglehub

from sklearn.preprocessing import StandardScaler

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.
Loaded PyTorch TimesFM, likely because python version is 3.11.11 (main, Dec  4 2024, 08:55:07) [GCC 11.4.0].


###Dataset Creation

In [5]:
class TimeSeriesDataset(Dataset):
  """Dataset for time series data compatible with TimesFM."""

  def __init__(self,
               series: np.ndarray,
               context_length: int,
               horizon_length: int,
               freq_type: int = 0):
    """
        Initialize dataset.

        Args:
            series: Time series data
            context_length: Number of past timesteps to use as input
            horizon_length: Number of future timesteps to predict
            freq_type: Frequency type (0, 1, or 2)
        """
    if freq_type not in [0, 1, 2]:
      raise ValueError("freq_type must be 0, 1, or 2")

    self.series = series
    self.context_length = context_length
    self.horizon_length = horizon_length
    self.freq_type = freq_type
    self._prepare_samples()

  def _prepare_samples(self) -> None:
    """Prepare sliding window samples from the time series."""
    self.samples = []
    total_length = self.context_length + self.horizon_length

    for start_idx in range(0, len(self.series) - total_length + 1):
      end_idx = start_idx + self.context_length
      x_context = self.series[start_idx:end_idx]
      x_future = self.series[end_idx:end_idx + self.horizon_length]
      self.samples.append((x_context, x_future))

  def __len__(self) -> int:
    return len(self.samples)

  def __getitem__(
      self, index: int
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    x_context, x_future = self.samples[index]

    x_context = torch.tensor(x_context, dtype=torch.float32)
    x_future = torch.tensor(x_future, dtype=torch.float32)

    input_padding = torch.zeros_like(x_context)
    freq = torch.tensor([self.freq_type], dtype=torch.long)

    return x_context, input_padding, freq, x_future

def prepare_datasets(series: np.ndarray,
                     context_length: int,
                     horizon_length: int,
                     freq_type: int = 0,
                     train_split: float = 0.8) -> Tuple[Dataset, Dataset]:
  """
    Prepare training and validation datasets from time series data.

    Args:
        series: Input time series data
        context_length: Number of past timesteps to use
        horizon_length: Number of future timesteps to predict
        freq_type: Frequency type (0, 1, or 2)
        train_split: Fraction of data to use for training

    Returns:
        Tuple of (train_dataset, val_dataset)
    """
  train_size = int(len(series) * train_split)
  train_data = series[:train_size]
  val_data = series[train_size:]

  # Create datasets with specified frequency type
  train_dataset = TimeSeriesDataset(train_data,
                                    context_length=context_length,
                                    horizon_length=horizon_length,
                                    freq_type=freq_type)

  val_dataset = TimeSeriesDataset(val_data,
                                  context_length=context_length,
                                  horizon_length=horizon_length,
                                  freq_type=freq_type)

  return train_dataset, val_dataset

# Modify method to include time_series of your choosing
def get_data(time_series: ArrayLike,
             context_len: int,
             horizon_len: int,
             freq_type: int = 0,
             train_split: float = 0.6) -> Tuple[Dataset, Dataset]:
    # Determine the training split index.
    train_size = int(len(time_series) * train_split)
    train_series = time_series[:train_size].reshape(-1, 1)

    # Fit a StandardScaler on the training data.
    scaler = StandardScaler()
    scaler.fit(train_series)

    # Transform the entire time series.
    normalized_series = scaler.transform(time_series.reshape(-1, 1)).flatten()

    # Create datasets using the normalized series.
    train_dataset, val_dataset = prepare_datasets(
        series=normalized_series,
        context_length=context_len,
        horizon_length=horizon_len,
        freq_type=freq_type,
        train_split=train_split,
    )

    print(f"Created datasets:")
    print(f"- Training samples: {len(train_dataset)}")
    print(f"- Validation samples: {len(val_dataset)}")
    print(f"- Using frequency type: {freq_type}")

    # Optionally, you can return the scaler as well if you want to invert the normalization later.
    return train_dataset, val_dataset

#### Get Weather data from albanian city.

In [6]:
import pandas as pd
from numpy.typing import ArrayLike
import kagglehub
import requests
from datetime import datetime, timedelta
from io import StringIO

def get_weather_data(city: str) -> ArrayLike:
    path = kagglehub.dataset_download("gucci1337/weather-of-albania-last-three-years")
    years = [2021, 2022, 2023]
    data_frames = []

    for year in years:
        file_path = f"{path}/data_weather/{city}/{city}{year}.csv"
        df = pd.read_csv(file_path)
        df = df.dropna(subset=['tavg'])  # Remove rows where 'tavg' is NaN
        data_frames.append(df['tavg'])

    # Concatenate the 'tavg' columns from each year's DataFrame
    concatenated_data = pd.concat(data_frames, ignore_index=True)

    return concatenated_data.values

def get_finance_data():
    """
    Returns a numpy array containing the finance data.
    """
    CSV_FILE_ABSOLUTE_PATH = "/content/AMZN-stock-price.csv"
    df = pd.read_csv(CSV_FILE_ABSOLUTE_PATH)
    # Assuming the second column holds the desired data.
    df = df.iloc[:, 1]
    return df.values

# energy
def get_energy_data(year: int):
    """
    Downloads daily consumption data for a given year, concatenates all days into a single DataFrame,
    and performs basic preprocessing (e.g., dropping NaN values).
    """
    all_data = []
    current_date = datetime(year, 1, 1)
    end_date = datetime(year, 12, 31)
    while current_date <= end_date:
        date_str = f"{current_date.day}.{current_date.month}.{current_date.year}"
        url = f"https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t{date_str}"
        print(f"Fetching data for {date_str} from:\n{url}")
        try:
            response = requests.get(url)
            response.raise_for_status()  # Raise an error for bad status codes
            daily_df = pd.read_csv(StringIO(response.text), sep=';')
            all_data.append(daily_df)
        except Exception as e:
            print(f"Error fetching data for {date_str}: {e}")
        current_date += timedelta(days=1)
    combined_data = pd.concat(all_data, ignore_index=True)
    string_values = combined_data.iloc[:, 1].values
    values_float = np.array([float(w.replace(',', '')) for w in string_values])
    return values_float



def get_new_cases_by_country(df: pd.DataFrame) -> dict:
    """
    Groups the DataFrame by 'Country', sorts by 'Date_reported', and returns a dictionary
    mapping country names to a NumPy array of new cases.
    """
    result = {}
    for country, group in df.groupby('Country'):
        group_sorted = group.sort_values('Date_reported')
        new_cases_array = group_sorted['New_cases'].to_numpy()
        result[country] = new_cases_array
    return result

def get_healthcare_data(country: str) -> ArrayLike:
    """
    Returns a NumPy array of new COVID-19 cases for the specified country.
    Data is taken from the WHO global daily dataset, and a slice [200:1200] is returned.
    """
    CSV_FILE_ABSOLUTE_PATH = "/content/WHO-COVID-19-global-daily-data.csv"
    df = pd.read_csv(CSV_FILE_ABSOLUTE_PATH)
    df = df.sort_values(['Country', 'Date_reported'])
    df['New_cases'] = df.groupby('Country')['New_cases'].transform(lambda group: group.interpolate(method='linear'))
    cases_dict = get_new_cases_by_country(df)
    new_cases = cases_dict.get(country)
    if new_cases is None:
        raise ValueError(f"Country '{country}' not found in dataset")
    return new_cases[200:1200]

###Model Creation

In [7]:
def get_model(load_weights: bool,
              per_core_batch_size: int,
              horizon_len: int,
              context_len: int):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    repo_id = "google/timesfm-2.0-500m-pytorch"
    hparams = TimesFmHparams(
        backend=device,
        per_core_batch_size=per_core_batch_size,
        horizon_len=horizon_len,
        num_layers=50,
        use_positional_embedding=False,
        context_len=context_len
    )

    tfm = TimesFm(hparams=hparams,
                  checkpoint=TimesFmCheckpoint(huggingface_repo_id=repo_id))

    model = PatchedTimeSeriesDecoder(tfm._model_config)
    if load_weights:
        checkpoint_path = path.join(snapshot_download(repo_id), "torch_model.ckpt")
        loaded_checkpoint = torch.load(checkpoint_path, weights_only=True)
        model.load_state_dict(loaded_checkpoint)
    # Return the model, hparams, configuration, and the underlying TimesFm object.
    return model, hparams, tfm._model_config, tfm

###Plotting Method

In [8]:
def plot_predictions(
    model: TimesFm,
    val_dataset: Dataset,
    save_path: Optional[str] = "predictions.png",
) -> None:
  """
    Plot model predictions against ground truth for a batch of validation data.

    Args:
      model: Trained TimesFM model
      val_dataset: Validation dataset
      save_path: Path to save the plot
    """
  import matplotlib.pyplot as plt

  model.eval()

  x_context, x_padding, freq, x_future = val_dataset[0]
  x_context = x_context.unsqueeze(0)  # Add batch dimension
  x_padding = x_padding.unsqueeze(0)
  freq = freq.unsqueeze(0)
  x_future = x_future.unsqueeze(0)

  device = next(model.parameters()).device
  x_context = x_context.to(device)
  x_padding = x_padding.to(device)
  freq = freq.to(device)
  x_future = x_future.to(device)

  with torch.no_grad():
    predictions = model(x_context, x_padding.float(), freq)
    predictions_mean = predictions[..., 0]  # [B, N, horizon_len]
    last_patch_pred = predictions_mean[:, -1, :]  # [B, horizon_len]

  context_vals = x_context[0].cpu().numpy()
  future_vals = x_future[0].cpu().numpy()
  pred_vals = last_patch_pred[0].cpu().numpy()

  context_len = len(context_vals)
  horizon_len = len(future_vals)

  plt.figure(figsize=(12, 6))

  plt.plot(range(context_len),
           context_vals,
           label="Historical Data",
           color="blue",
           linewidth=2)

  plt.plot(
      range(context_len, context_len + horizon_len),
      future_vals,
      label="Ground Truth",
      color="green",
      linestyle="--",
      linewidth=2,
  )

  plt.plot(range(context_len, context_len + horizon_len),
           pred_vals,
           label="Prediction",
           color="red",
           linewidth=2)

  plt.xlabel("Time Step")
  plt.ylabel("Value")
  plt.title("TimesFM Predictions vs Ground Truth")
  plt.legend()
  plt.grid(True)

  if save_path:
    plt.savefig(save_path)
    print(f"Plot saved to {save_path}")

  plt.close()

###Fine-tuning on Single GPU

In [9]:
import numpy as np
import torch
import random
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score, explained_variance_score

# Define the configuration pairs: (context_len, horizon_len)
config_pairs = [(64, 32), (128, 64), (256, 128)]
batch_size = 64
train_split = 0.6
freq_type = 1  # using medium frequency indicator
num_samples = 10  # Number of samples to evaluate on

def run_inference_for_dataset(dataset_name: str, time_series: np.ndarray):
    print(f"\n===== Dataset: {dataset_name} =====")
    # Compute naive scale for MASE using the training portion of the series.
    insample_data = time_series[:int(len(time_series) * train_split)]
    naive_errors = np.abs(insample_data[1:] - insample_data[:-1])
    scale = np.mean(naive_errors) if naive_errors.size > 0 else 1e-8

    # Loop over each configuration pair.
    for idx, (context_len, horizon_len) in enumerate(config_pairs, start=1):
        print(f"\n--- Experiment {idx}: context_len={context_len}, horizon_len={horizon_len} ---")

        # Load the model and its associated TimesFM object.
        # Note: Here, get_model is assumed to return (model, hparams, tfm_config, tfm)
        model, hparams, tfm_config, tfm = get_model(
            load_weights=True,
            per_core_batch_size=batch_size,
            horizon_len=horizon_len,
            context_len=context_len
        )

        # Prepare training and validation datasets.
        train_dataset, val_dataset = get_data(
            time_series,
            context_len=context_len,
            horizon_len=horizon_len,
            freq_type=freq_type,
            train_split=train_split
        )

        # Randomly sample a subset of the validation dataset.
        total_val_samples = len(val_dataset)
        if total_val_samples > num_samples:
            sampled_indices = random.sample(range(total_val_samples), num_samples)
        else:
            sampled_indices = list(range(total_val_samples))
            print(f"Note: Validation dataset only contains {total_val_samples} samples.")

        # Initialize lists to accumulate forecasts and ground truth values.
        all_preds = []
        all_targets = []

        # Loop over the sampled indices.
        for i in sampled_indices:
            sample = val_dataset[i]
            x_context, x_padding, freq, x_future = sample
            # Convert context to numpy and extract frequency indicator.
            context_np = x_context.cpu().numpy()
            freq_val = int(freq.item())

            # Use the TimesFM forecast API.
            point_forecast, _ = tfm.forecast([context_np], freq=[freq_val])
            pred = point_forecast[0]  # Forecast output (expected shape: [horizon_len])
            true_vals = x_future.cpu().numpy()  # Ground truth

            all_preds.append(pred)
            all_targets.append(true_vals)

        # Concatenate predictions and ground truth arrays.
        all_preds = np.concatenate(all_preds)
        all_targets = np.concatenate(all_targets)

        # Compute custom metrics.
        custom_mae = np.mean(np.abs(all_preds - all_targets))
        custom_rmse = np.sqrt(np.mean((all_preds - all_targets) ** 2))
        custom_mase = custom_mae / scale

        # Compute additional metrics using sklearn.
        sklearn_rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
        sklearn_mae = mean_absolute_error(all_targets, all_preds)
        sklearn_mape = mean_absolute_percentage_error(all_targets, all_preds)
        r2 = r2_score(all_targets, all_preds)
        explained_var = explained_variance_score(all_targets, all_preds)

        print(f"Custom Metrics -> MAE: {custom_mae:.4f}, RMSE: {custom_rmse:.4f}, MASE: {custom_mase:.4f}")
        print("Sklearn Metrics:")
        print(f"  RMSE: {sklearn_rmse:.4f}")
        print(f"  MAE: {sklearn_mae:.4f}")
        print(f"  MAPE: {sklearn_mape:.4f}")
        print(f"  R^2: {r2:.4f}")
        print(f"  Explained Variance: {explained_var:.4f}")
        print("-" * 60)


# Fetch data for each dataset.
weather_series = get_weather_data("lezhe")
finance_series = get_finance_data()
energy_series = get_energy_data(2024)
# For healthcare data, adjust parameters as needed (e.g., country name).
healthcare_series = get_healthcare_data("Germany")

# Run inference for each dataset.
run_inference_for_dataset("weather", weather_series)
run_inference_for_dataset("finance", finance_series)
run_inference_for_dataset("energy", energy_series)
run_inference_for_dataset("healthcare", healthcare_series)


Fetching data for 1.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t1.1.2024
Fetching data for 2.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t2.1.2024
Fetching data for 3.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t3.1.2024
Fetching data for 4.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t4.1.2024
Fetching data for 5.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t5.1.2024
Fetching data for 6.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t6.1.2024
Fetching data for 7.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t7.1.2024
Fetching data for 8.1.2024 from:
https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t8.1.2024
Fetching data for 9.1.2024 from:
https://www.eview.de/e1/p3Export.php?fr

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Created datasets:
- Training samples: 562
- Validation samples: 343
- Using frequency type: 1
Custom Metrics -> MAE: 0.3954, RMSE: 0.5067, MASE: 0.3552
Sklearn Metrics:
  RMSE: 0.5067
  MAE: 0.3954
  MAPE: 1.7345
  R^2: 0.6048
  Explained Variance: 0.6121
------------------------------------------------------------

--- Experiment 2: context_len=128, horizon_len=64 ---


...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Created datasets:
- Training samples: 409
- Validation samples: 209
- Using frequency type: 1
Custom Metrics -> MAE: 0.5296, RMSE: 0.6621, MASE: 0.0001
Sklearn Metrics:
  RMSE: 0.6621
  MAE: 0.5296
  MAPE: 10.9817
  R^2: -1.0936
  Explained Variance: -0.9151
------------------------------------------------------------

--- Experiment 3: context_len=256, horizon_len=128 ---


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Created datasets:
- Training samples: 217
- Validation samples: 17
- Using frequency type: 1
Custom Metrics -> MAE: 0.6699, RMSE: 0.8237, MASE: 0.0001
Sklearn Metrics:
  RMSE: 0.8237
  MAE: 0.6699
  MAPE: 3.8703
  R^2: -16.4243
  Explained Variance: -7.5461
------------------------------------------------------------


###Show the directory where the graph is created

In [10]:
!ls

datasets  experiments  notebooks  poetry.lock	  README.md  tests
docs	  LICENSE      peft	  pyproject.toml  src


###Freeing Up GPU Resources

In [11]:
from numba import cuda
device = cuda.get_current_device()
device.reset()
device = cuda.get_current_device()